# WM-811K 취약 클래스 개선 실험

동일한 lot 비중복 분할에서 세 학습 전략을 비교합니다. 모델 선택은 **validation macro-F1만** 사용하며 test는 선택이 끝난 뒤 참고용으로 비교합니다. 모든 전략은 train에서만 회전·반전 증강을 적용합니다.

In [ ]:
!pip install -q pandas==2.2.3 scikit-learn matplotlib

from google.colab import drive, files
from pathlib import Path
from datetime import datetime
import json, shutil, subprocess, sys
import matplotlib.pyplot as plt
import pandas as pd
import torch

drive.mount('/content/drive')
assert torch.cuda.is_available(), '런타임 유형을 T4 GPU로 바꾼 뒤 다시 실행하세요.'
print('GPU:', torch.cuda.get_device_name(0))

DRIVE_DATA_DIR = Path('/content/drive/MyDrive/AI_project/WM811K_processed')
LOCAL_DATA_DIR = Path('/content/wm811k_processed')
LOCAL_DATA_DIR.mkdir(parents=True, exist_ok=True)
for name in ('wafer_maps_64.npy', 'labels.npy'):
    source = DRIVE_DATA_DIR / name
    assert source.is_file(), f'전처리 출력 파일이 없습니다: {source}'
    destination = LOCAL_DATA_DIR / name
    if not destination.is_file() or destination.stat().st_size != source.stat().st_size:
        print('로컬 SSD로 복사:', name)
        shutil.copy2(source, destination)
print('학습 데이터 준비 완료:', LOCAL_DATA_DIR)

## 로컬 프로젝트 파일 두 개 업로드

첫 창에서 `코드/wm811k/train_wm811k_cnn.py`, 두 번째 창에서 `결과물/wm811k/split_results/split_assignments.csv`를 선택합니다.

In [ ]:
uploaded = {}
print('1/2: train_wm811k_cnn.py를 선택하세요.')
uploaded.update(files.upload())
print('2/2: split_assignments.csv를 선택하세요.')
uploaded.update(files.upload())

def uploaded_bytes(prefix, suffix):
    matches = [name for name in uploaded if name.startswith(prefix) and name.endswith(suffix)]
    assert matches, f'선택하지 않은 파일: {prefix}{suffix}'
    return uploaded[matches[-1]]

TRAIN_SCRIPT = Path('/content/train_wm811k_cnn.py')
ASSIGNMENTS = Path('/content/split_assignments.csv')
TRAIN_SCRIPT.write_bytes(uploaded_bytes('train_wm811k_cnn', '.py'))
ASSIGNMENTS.write_bytes(uploaded_bytes('split_assignments', '.csv'))
print('업로드 완료')

## 세 전략 학습

약 20~30분이 예상됩니다. `focal + shuffle`, `weighted CE + shuffle`, `일반 CE + 제곱근 균형 샘플링`을 같은 조건으로 실행합니다.

In [ ]:
RUN_ID = datetime.now().strftime('%Y%m%d_%H%M%S')
RUN_DIR = Path('/content/drive/MyDrive/AI_project/WM811K_training') / f'strategy_comparison_{RUN_ID}'
RUN_DIR.mkdir(parents=True, exist_ok=False)
EXPERIMENTS = [
    {'name': 'focal_shuffle', 'loss': 'focal', 'sampling': 'shuffle'},
    {'name': 'weighted_ce_shuffle', 'loss': 'weighted_ce', 'sampling': 'shuffle'},
    {'name': 'ce_sqrt_balanced', 'loss': 'cross_entropy', 'sampling': 'sqrt_balanced'},
]

for experiment in EXPERIMENTS:
    output_dir = RUN_DIR / experiment['name']
    print('\n' + '=' * 72)
    print('학습 시작:', experiment)
    command = [
        sys.executable, '-u', str(TRAIN_SCRIPT),
        '--maps', str(LOCAL_DATA_DIR / 'wafer_maps_64.npy'),
        '--labels', str(LOCAL_DATA_DIR / 'labels.npy'),
        '--assignments', str(ASSIGNMENTS),
        '--output-dir', str(output_dir),
        '--epochs', '20',
        '--batch-size', '256',
        '--num-workers', '2',
        '--loss', experiment['loss'],
        '--sampling', experiment['sampling'],
        '--sampling-max-multiplier', '8',
        '--seed', '42',
    ]
    subprocess.run(command, check=True)
print('세 전략 학습 완료:', RUN_DIR)

## Validation으로 모델 선택 고정

아래 셀은 validation macro-F1과 균형 정확도로만 모델을 선택하고 그 결정을 JSON에 먼저 기록합니다. test 비교는 선택 결과를 바꾸지 않습니다.

In [ ]:
validation_rows = []
summaries = {}
for experiment in EXPERIMENTS:
    name = experiment['name']
    result_dir = RUN_DIR / name
    summary = json.loads((result_dir / 'run_summary.json').read_text(encoding='utf-8'))
    summaries[name] = summary
    report = pd.read_csv(result_dir / 'validation_classification_report.csv', index_col=0)
    validation_rows.append({
        'strategy': name,
        'validation_macro_f1': summary['validation_metrics']['macro_f1'],
        'validation_balanced_accuracy': summary['validation_metrics']['balanced_accuracy'],
        'validation_accuracy': summary['validation_metrics']['accuracy'],
        'scratch_recall': report.loc['Scratch', 'recall'],
        'loc_recall': report.loc['Loc', 'recall'],
        'near_full_recall': report.loc['Near-full', 'recall'],
        'best_epoch': summary['best_epoch'],
    })
validation_comparison = pd.DataFrame(validation_rows).sort_values(
    ['validation_macro_f1', 'validation_balanced_accuracy'], ascending=False
)
selected_strategy = validation_comparison.iloc[0]['strategy']
selection = {
    'selected_strategy': selected_strategy,
    'primary_metric': 'validation_macro_f1',
    'tie_breaker': 'validation_balanced_accuracy',
    'selection_uses_test': False,
    'random_state': 42,
}
validation_comparison.to_csv(RUN_DIR / 'validation_strategy_comparison.csv', index=False)
(RUN_DIR / 'selection_protocol.json').write_text(
    json.dumps(selection, ensure_ascii=False, indent=2), encoding='utf-8'
)
shutil.copy2(RUN_DIR / selected_strategy / 'best_model.pt', RUN_DIR / 'selected_model.pt')
print('VALIDATION 선택:', json.dumps(selection, ensure_ascii=False))
display(validation_comparison)

test_comparison = pd.DataFrame([
    {
        'strategy': name,
        'test_macro_f1_exploratory': summary['test_metrics']['macro_f1'],
        'test_balanced_accuracy_exploratory': summary['test_metrics']['balanced_accuracy'],
        'test_accuracy_exploratory': summary['test_metrics']['accuracy'],
        'selected_on_validation': name == selected_strategy,
    }
    for name, summary in summaries.items()
]).sort_values('test_macro_f1_exploratory', ascending=False)
test_comparison.to_csv(RUN_DIR / 'test_strategy_comparison_exploratory.csv', index=False)
display(test_comparison)

axis = validation_comparison.set_index('strategy')[
    ['validation_macro_f1', 'scratch_recall', 'loc_recall']
].plot(kind='bar', figsize=(10, 5), ylim=(0, 1), rot=0)
axis.set_title('Validation-only strategy comparison')
axis.set_ylabel('Score')
axis.figure.tight_layout()
axis.figure.savefig(RUN_DIR / 'validation_strategy_comparison.png', dpi=180)
plt.show()

## 결과 ZIP 다운로드

다운로드한 `wm811k_strategy_comparison_results.zip`을 로컬 프로젝트의 `결과물/wm811k/colab_가져오기/`에 압축을 풀지 않고 넣습니다.

In [ ]:
ZIP_BASE = Path('/content/wm811k_strategy_comparison_results')
zip_path = Path(shutil.make_archive(str(ZIP_BASE), 'zip', root_dir=RUN_DIR))
print('ZIP:', zip_path, f'{zip_path.stat().st_size / 1024**2:.1f} MB')
files.download(str(zip_path))